# 01 — The Strict API Workflow: Works Until It Doesn't

**Goal:** Build a hard-coded API call chain that answers the question  
*"Is my package going to arrive on time?"*

We'll see it succeed on the **happy path**, then watch it **silently fail** on the edge case — the one account with two shipments and a disrupted carrier.

---

## The Architecture We're Testing

```
┌─────────────────────────────────────────────────────────────────┐
│                   STRICT API CHAIN                              │
│                                                                 │
│  Input: account_id                                              │
│     ↓                                                           │
│  GET /accounts/{id}             (always exactly one result)     │
│     ↓                                                           │
│  GET /shipments?account_id={id} (assume shipments[0])  ← BUG   │
│     ↓                                                           │
│  GET /weather/{destination}     (only destination)    ← BUG    │
│     ↓                                                           │
│  GET /carriers/{carrier_id}                                     │
│     ↓                                                           │
│  return hard-coded verdict string                               │
└─────────────────────────────────────────────────────────────────┘
```

This is typical of agent pipelines built with LangChain `SequentialChain`, crewAI rigid crews, or hand-written orchestration code.

In [ ]:
import os, sys
os.chdir(os.path.abspath(".."))  # set project root

import requests
from rich import print as rprint
from rich.panel import Panel
from rich.console import Console

console = Console()

LOGISTICS = "http://localhost:8001"
WEATHER   = "http://localhost:8002"

print("Libraries loaded. Checking servers...")
assert requests.get(f"{LOGISTICS}/").status_code == 200, "Logistics API not running!"
assert requests.get(f"{WEATHER}/").status_code == 200, "Weather API not running!"
print("✅ Both servers are reachable.")

## Part 1 — Build the Strict Agent

This agent is a function: `check_shipment_arrival(account_id)`.  
It calls APIs in a fixed sequence with **no dynamic branching**.

In [ ]:
def strict_shipment_agent(account_id: str) -> str:
    """
    Rigid API chain agent.
    
    ASSUMPTION: Each account has exactly ONE shipment.
    ASSUMPTION: We only check destination weather.
    ASSUMPTION: If carrier status != 'on_time', it must be 'delayed'.
    
    These assumptions are typical of first-draft pipeline code.
    They work until the data doesn't match the assumptions.
    """
    trace = []  # We'll log each step for analysis
    
    # ── Step 1: Get account ───────────────────────────────────────────────────
    trace.append(f"[Step 1] GET /accounts/{account_id}")
    account = requests.get(f"{LOGISTICS}/accounts/{account_id}").json()
    trace.append(f"  → {account['name']} ({account['tier']}) in {account['home_city']}")
    
    # ── Step 2: Get shipments ─────────────────────────────────────────────────
    # BUG #1: We blindly take shipments[0]. If there are multiple, we miss them.
    trace.append(f"[Step 2] GET /shipments?account_id={account_id}")
    shipments = requests.get(f"{LOGISTICS}/shipments", params={"account_id": account_id}).json()
    trace.append(f"  → Found {len(shipments)} shipment(s). Taking first only.")
    
    shipment = shipments[0]  # ← BUG: silently ignores any others
    trace.append(f"  → Using: {shipment['shipment_id']} → {shipment['destination_city']}")
    
    # ── Step 3: Get weather (destination ONLY) ────────────────────────────────
    # BUG #2: We only check destination. Origin weather is ignored.
    dest = shipment["destination_city"]
    trace.append(f"[Step 3] GET /weather/{dest}")
    weather = requests.get(f"{WEATHER}/weather/{dest}").json()
    trace.append(f"  → {dest}: {weather['condition']} (risk: {weather['risk_level']})")
    
    # ── Step 4: Get carrier status ────────────────────────────────────────────
    carrier_id = shipment["carrier_id"]
    trace.append(f"[Step 4] GET /carriers/{carrier_id}")
    carrier = requests.get(f"{LOGISTICS}/carriers/{carrier_id}").json()
    trace.append(f"  → {carrier['name']}: status={carrier['status']}")
    
    # ── Step 5: Build verdict ─────────────────────────────────────────────────
    # BUG #3: Rigid conditionals. 'disrupted' falls through to the generic branch.
    if carrier["status"] == "on_time" and weather["risk_level"] == "low":
        verdict = f"✅ Your shipment (ETA: {shipment['eta_date']}) is on track."
    elif weather["risk_level"] in ("medium", "high"):
        verdict = f"⚠️ Weather delays possible. ETA {shipment['eta_date']} may slip."
    else:
        # This branch catches 'delayed' AND 'disrupted' — same message, wrong severity
        verdict = f"⚠️ Carrier reports delays. ETA {shipment['eta_date']} may slip."
    
    trace.append(f"[Verdict] {verdict}")
    
    return "\n".join(trace)

## Part 2 — Happy Path (ACC-001: Single Shipment, No Issues)

In [ ]:
console.print(Panel(
    strict_shipment_agent("ACC-001"),
    title="[bold green]Happy Path — ACC-001 (Single Shipment)[/bold green]",
    border_style="green"
))

✅ **It works!** One account, one shipment, on-time carrier, good weather. The rigid chain handles this perfectly.

---

## Part 3 — The Edge Case (ACC-002: Two Shipments, Disrupted Carrier)

Now let's run the same agent against the problematic account.

In [ ]:
console.print(Panel(
    strict_shipment_agent("ACC-002"),
    title="[bold red]Edge Case — ACC-002 (Two Shipments, CARR-B Disrupted)[/bold red]",
    border_style="red"
))

## Part 4 — Documenting the Failures

Let's be precise about what went wrong and why.

In [ ]:
from rich.table import Table

# What the agent reported vs. what it SHOULD have reported
shipments = requests.get(f"{LOGISTICS}/shipments", params={"account_id": "ACC-002"}).json()
carrier_b = requests.get(f"{LOGISTICS}/carriers/CARR-B").json()
denver_wx = requests.get(f"{WEATHER}/weather/Denver").json()
miami_wx  = requests.get(f"{WEATHER}/weather/Miami").json()

table = Table(title="Failures in the Strict API Chain", show_lines=True)
table.add_column("Failure",     style="bold red",   width=20)
table.add_column("What Happened",                   width=35)
table.add_column("What Should Happen",              width=35)

table.add_row(
    "#1 — Missing Shipment",
    f"Checked SHP-101 (Miami) only.\nSHP-102 (Denver) was silently dropped.",
    f"Process BOTH shipments:\n• SHP-101 → Miami (low risk)\n• SHP-102 → Denver (CRITICAL)"
)
table.add_row(
    "#2 — Wrong Severity",
    f"CARR-B is 'disrupted' (+48h).\nAgent said 'delays possible' — same\nmessage as a minor delay.",
    f"Disrupted ≠ delayed.\nSHP-102 is likely to miss ETA by\n{carrier_b['delay_hours']}h. Flag as CRITICAL."
)
table.add_row(
    "#3 — Weather Blindspot",
    f"Only checked destination weather.\nDenver blizzard not surfaced\nuntil SHP-102 accidentally checked.",
    f"Check BOTH origin and destination.\nDenver: '{denver_wx['condition']}'\nRisk: {denver_wx['risk_level'].upper()}"
)
table.add_row(
    "#4 — No Corroboration",
    "Carrier data and weather data\nnever cross-referenced.",
    "CARR-B disrupted + Denver blizzard\n= corroborating evidence = higher\nconfidence in bad outcome."
)

console.print(table)

## Part 5 — Attempting a Patch

Can we fix this with more conditionals? Let's try — and watch the complexity explode.

In [ ]:
def strict_shipment_agent_v2(account_id: str) -> str:
    """
    Patched version — v2.
    We added a loop for multiple shipments and better carrier handling.
    But look how quickly this becomes unmaintainable...
    """
    account   = requests.get(f"{LOGISTICS}/accounts/{account_id}").json()
    shipments = requests.get(f"{LOGISTICS}/shipments", params={"account_id": account_id}).json()
    
    results = []
    
    for shipment in shipments:  # ← PATCH 1: loop added
        dest     = shipment["destination_city"]
        origin   = shipment["origin_city"]
        carrier  = requests.get(f"{LOGISTICS}/carriers/{shipment['carrier_id']}").json()
        dest_wx  = requests.get(f"{WEATHER}/weather/{dest}").json()
        orig_wx  = requests.get(f"{WEATHER}/weather/{origin}").json()  # ← PATCH 2
        
        # PATCH 3: explicit disrupted handling (but now we need more branches)
        if carrier["status"] == "disrupted" and dest_wx["risk_level"] == "high":
            verdict = f"🚨 CRITICAL: {shipment['shipment_id']} to {dest} — carrier disrupted AND weather high risk."
        elif carrier["status"] == "disrupted":
            verdict = f"🚨 HIGH RISK: {shipment['shipment_id']} — carrier disrupted (+{carrier['delay_hours']}h)."
        elif dest_wx["risk_level"] == "high":
            verdict = f"⚠️ HIGH: {shipment['shipment_id']} to {dest} — severe weather at destination."
        elif dest_wx["risk_level"] == "medium" or orig_wx["risk_level"] in ("medium", "high"):
            verdict = f"⚠️ MEDIUM: {shipment['shipment_id']} — weather caution."
        elif carrier["status"] == "delayed":
            verdict = f"⚠️ DELAYED: {shipment['shipment_id']} — minor carrier delay."
        elif shipment["status"] == "held":
            verdict = f"🔒 HELD: {shipment['shipment_id']} — shipment is on hold."
        # What about: disrupted + medium weather? held + disrupted? two shipments to same carrier?
        # Every new edge case needs another branch. This doesn't scale.
        else:
            verdict = f"✅ ON TRACK: {shipment['shipment_id']} (ETA: {shipment['eta_date']})."
        
        results.append(verdict)
    
    return "\n".join(results)


console.print(Panel(
    strict_shipment_agent_v2("ACC-002"),
    title="[bold yellow]Patched v2 — ACC-002[/bold yellow]",
    border_style="yellow"
))

console.print("\n[bold]Observation:[/bold] v2 catches the case — but look at the code.")
console.print("• [yellow]6 conditional branches[/yellow] for 4 data fields.")
console.print("• [yellow]Each new scenario requires a new elif.[/yellow]")
console.print("• [yellow]No reasoning — just pattern matching against known cases.[/yellow]")
console.print("• [yellow]Still can't explain WHY or suggest alternatives.[/yellow]")

## Part 6 — Measuring the Chain

Let's record baseline metrics to compare against the MCP agent in notebook 03.

In [ ]:
import time

# Measure latency for both accounts
timings = {}
for acc_id in ["ACC-001", "ACC-002"]:
    start = time.perf_counter()
    strict_shipment_agent_v2(acc_id)
    elapsed = time.perf_counter() - start
    timings[acc_id] = round(elapsed, 3)

print("API Chain Latency (no LLM calls):")
for acc_id, t in timings.items():
    print(f"  {acc_id}: {t}s")

# Count total API calls made
print("\nAPI Calls (v2 patched chain):")
print("  ACC-001 (1 shipment): 1 account + 1 shipment list + 1 carrier + 2 weather = 5 calls")
print("  ACC-002 (2 shipments): 1 account + 1 shipment list + 2 carriers + 4 weather = 8 calls")
print("  Note: Token cost = 0 (no LLM used in this chain)")

# Save metrics for notebook 03
import json
metrics = {
    "api_chain_latency": timings,
    "api_chain_calls": {"ACC-001": 5, "ACC-002": 8},
    "api_chain_tokens": 0,
    "api_chain_handles_edge_case": False,
    "api_chain_reasoning": False,
}
with open("../api_chain_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\n✅ Metrics saved to api_chain_metrics.json")

## Summary — Why the Strict Chain Fails

| Problem | Root Cause | Patchable? |
|---------|-----------|------------|
| Silently drops 2nd shipment | `shipments[0]` assumption | Yes, but requires a loop |
| Wrong severity for disrupted carrier | Flat conditional | Yes, more elif branches |
| Misses origin weather | Hard-coded "destination only" | Yes, another API call |
| No cross-referencing | No reasoning layer | **No** — requires LLM |
| Can't explain WHY | No natural language generation | **No** — requires LLM |
| Breaks on unknown edge cases | All cases must be pre-coded | **No** — requires dynamic reasoning |

**The last two rows cannot be fixed with more `elif` statements.** They require a model that reasons. That's Notebook 02.

---
**Next → `02_mcp_react_workflow.ipynb`**